# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: NB03 will be used to create all of our data visualisations that will display safe routes home. 
</div>

## Data Visualisation Pipeline

1. Understand the data 
   - Do some quick checks of the database, so I can see what I am working with
2. Create a scoring system
   - ss
3. Test the usability of the scoring system
   - ss
4. Create plots and maps
   - ss

In [2]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

## Step 1 - Understand the Database


In [3]:
DB_PATH = Path("data/processed/nightclub_safety.db")
conn = sqlite3.connect(str(DB_PATH))

# List all tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print("Tables in database:")
display(tables)

# Inspect each table 
for table in tables["name"]:
    print(f"TABLE: {table}")

    # schema
    schema = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print("\nColumns & types:")
    display(schema[["name", "type"]])

    # row count
    count = pd.read_sql_query(f"SELECT COUNT(*) AS rows FROM {table};", conn)
    print("\nNumber of rows:")
    display(count)

    # preview data
    preview = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5;", conn)
    print("\nExample rows:")
    display(preview)

conn.close()


Tables in database:


,name
0,areas
1,sqlite_sequence
2,clubs
3,destinations
4,routes
5,crimes
6,route_crimes


TABLE: areas

Columns & types:


,name,type
0,area_id,INTEGER
1,name,TEXT



Number of rows:


,rows
0,3


... (34 outputs left out)


Number of rows:


,rows
0,4055



Example rows:


,route_id,crime_id
0,4,-1
1,4,0
2,4,1
3,4,2
4,4,3


## Step 2 - Create a Scoring System 


In [5]:
# ============================================================
# WITHIN-AREA SAFETY SCORING (routes) — FULL WORKING CODE
# - pulls routes for ONE area (e.g., Bermondsey / Dalston)
# - computes crimes per km + per min
# - min–max normalises WITHIN that area
# - computes weighted risk score -> safety score (0–100)
# - ranks routes overall + within each club
# - basic visualisations (hist + scatter)
# ============================================================

from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# 0) SETTINGS
# -------------------------
DB_PATH = Path("data/processed/nightclub_safety.db")
assert DB_PATH.exists(), f"DB not found: {DB_PATH.resolve()}"

AREA_NAME = "Bermondsey"         # <- change to "Dalston" when you want
ROUTE_CRIMES_TABLE = "route_crimes"  # <- change to "routes_crimes" if that's your actual table name

# weights (within-area)
W_CRIME_KM = 0.4
W_CRIME_MIN = 0.4
W_DIST = 0.1
W_DUR = 0.1


# -------------------------
# 1) CONNECT + EXTRACT ROUTE-LEVEL DATA FOR THIS AREA
# -------------------------
conn = sqlite3.connect(str(DB_PATH))

query = f"""
SELECT
    a.name AS area_name,

    c.club_id,
    -- NOTE: club name column varies by schema; if this errors, remove it or rename it
    c.displayName AS club_name,

    d.destination_id,
    d.displayName AS destination_name,

    r.route_id,
    r.distanceMeters  AS distance_m,
    r.durationSeconds AS duration_s,

    COUNT(rc.crime_id) AS crime_count,

    -- rate measures (guard against divide-by-zero)
    CASE
        WHEN r.distanceMeters > 0 THEN COUNT(rc.crime_id) / (r.distanceMeters / 1000.0)
        ELSE 0
    END AS crimes_per_km,

    CASE
        WHEN r.durationSeconds > 0 THEN COUNT(rc.crime_id) / (r.durationSeconds / 60.0)
        ELSE 0
    END AS crimes_per_min

FROM areas a
JOIN clubs c ON a.area_id = c.area_id
JOIN destinations d ON c.club_id = d.club_id
JOIN routes r ON d.destination_id = r.destination_id
LEFT JOIN {ROUTE_CRIMES_TABLE} rc ON r.route_id = rc.route_id
WHERE a.name = '{AREA_NAME}'
GROUP BY
    a.name,
    c.club_id, c.displayName,
    d.destination_id, d.displayName,
    r.route_id, r.distanceMeters, r.durationSeconds;
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f"Area: {AREA_NAME}")
print("Rows:", len(df), "| Unique routes:", df["route_id"].nunique(), "| Unique clubs:", df["club_id"].nunique())


# -------------------------
# 2) CLEAN + DERIVED COLUMNS
# -------------------------
# Ensure numeric
num_cols = ["distance_m", "duration_s", "crime_count", "crimes_per_km", "crimes_per_min"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df["crime_count"] = df["crime_count"].fillna(0).astype(int)
df["crimes_per_km"] = df["crimes_per_km"].fillna(0)
df["crimes_per_min"] = df["crimes_per_min"].fillna(0)

# Helpful units
df["distance_km"] = df["distance_m"] / 1000.0
df["duration_min"] = df["duration_s"] / 60.0


# -------------------------
# 3) WITHIN-AREA NORMALISATION
# -------------------------
def minmax(series: pd.Series) -> pd.Series:
    s = series.astype(float)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

df["crime_km_norm"] = minmax(df["crimes_per_km"])
df["crime_min_norm"] = minmax(df["crimes_per_min"])
df["dist_norm"] = minmax(df["distance_km"])
df["dur_norm"] = minmax(df["duration_min"])


# -------------------------
# 4) RISK SCORE -> SAFETY SCORE
# -------------------------
df["risk_score"] = (
    W_CRIME_KM  * df["crime_km_norm"] +
    W_CRIME_MIN * df["crime_min_norm"] +
    W_DIST      * df["dist_norm"] +
    W_DUR       * df["dur_norm"]
)

df["safety_score"] = 1 - df["risk_score"]
df["safety_score_100"] = (df["safety_score"] * 100).round(2)


# -------------------------
# 5) RANKING (OVERALL + WITHIN CLUB)
# -------------------------
df = df.sort_values(["safety_score_100", "crime_count"], ascending=[False, True]).copy()
df["rank_in_area"] = np.arange(1, len(df) + 1)

df["rank_within_club"] = (
    df.groupby("club_id")["safety_score_100"]
      .rank(method="dense", ascending=False)
      .astype(int)
)

print("\nTop 10 safest routes overall:")
display(df[[
    "rank_in_area", "club_id", "club_name", "destination_name",
    "distance_km", "duration_min", "crime_count",
    "crimes_per_km", "crimes_per_min", "safety_score_100"
]].head(10))

print("\nTop 3 safest routes per club:")
display(
    df[df["rank_within_club"] <= 3]
      .sort_values(["club_id", "rank_within_club"])[
          ["club_id", "club_name", "rank_within_club", "destination_name", "safety_score_100", "crime_count"]
      ]
)


# -------------------------
# 6) BASIC VISUALISATIONS (STEP 9 STYLE)
# -------------------------
plt.figure()
plt.hist(df["safety_score_100"].dropna(), bins=20)
plt.xlabel("Safety score (0–100)")
plt.ylabel("Number of routes")
plt.title(f"{AREA_NAME}: Safety score distribution")
plt.show()

plt.figure()
plt.scatter(df["distance_km"], df["safety_score_100"])
plt.xlabel("Distance (km)")
plt.ylabel("Safety score (0–100)")
plt.title(f"{AREA_NAME}: Safety vs distance")
plt.show()

plt.figure()
plt.scatter(df["duration_min"], df["safety_score_100"])
plt.xlabel("Duration (min)")
plt.ylabel("Safety score (0–100)")
plt.title(f"{AREA_NAME}: Safety vs duration")
plt.show()


# -------------------------
# 7) EXPORT (OPTIONAL)
# -------------------------
out_csv = f"{AREA_NAME.lower()}_within_area_route_scores.csv"
df.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


Area: Bermondsey
Rows: 68 | Unique routes: 68 | Unique clubs: 7


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64')